# 00 · Lift warm-up: dataset anatomy

**Purpose.** Look inside the robomimic Lift (proficient-human, `ph`) dataset before training
anything: what one demonstration contains, what an action means, and how simulator state,
observations and actions relate.

| | |
|---|---|
| Track | A (upstream data, inspection only) |
| Kernel | sim venv (`$WORK/franka-il/venvs/sim`); needs `h5py`, `numpy`, `matplotlib` |
| Inputs | `$HPCWORK/franka-il/data/robomimic_v15/lift/ph/{low_dim_v15,demo_v15}.hdf5` + `expected.sha256` (override the root with `FRANKA_DATA`) |
| Outputs | none written; figures inline only |
| Runtime | < 1 min on a login node, CPU only; §7 runs the simulator without rendering |

**Success checks** (each section ends with an `assert`):
1. Checksums match the pinned values.
2. 200 demos; `train`/`valid` = 180/20.
3. Timestep is 0.05 s (20 Hz) and `states` = `[time | qpos(16) | qvel(15)]`.
4. `next_obs[t] == obs[t+1]`; actions lie in `[-1, 1]`; gripper action is binary.

Questions this notebook answers: what does 20 Hz mean, what is `num_samples`, is this
time-series modelling, what are `qpos`/`qvel`, and what happens inside `env.step(action)`.

## 0. Setup and checksum

In [ ]:
import hashlib, json, os, xml.etree.ElementTree as ET
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np

root = Path(os.environ.get("FRANKA_DATA") or Path(os.environ["HPCWORK"]) / "franka-il/data/robomimic_v15")
ph = root / "lift" / "ph"
LOW_DIM, RAW = ph / "low_dim_v15.hdf5", ph / "demo_v15.hdf5"

def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        while block := fh.read(chunk):
            h.update(block)
    return h.hexdigest()

expected = dict(reversed(line.split()) for line in (ph / "expected.sha256").read_text().splitlines() if line.strip())
for name, digest in expected.items():
    ok = sha256(ph / name) == digest
    print(f"{name:20s} {'OK' if ok else 'MISMATCH'}")
    assert ok

## 1. File layout: a dataset of episodes

An HDF5 file is a folder tree of arrays. robomimic puts one **episode** (demonstration) per
group under `data/`, and named lists of episode IDs ("filter keys") under `mask/`.

In [ ]:
f = h5py.File(LOW_DIM, "r")
demos = sorted(f["data"].keys(), key=lambda k: int(k.split("_")[1]))
print("top level:", list(f.keys()))
print("data attrs:", list(f["data"].attrs.keys()), "| total transitions:", f["data"].attrs["total"])
print("demos:", len(demos), demos[:3], "...")
print("\nfilter keys (mask/):")
for k in f["mask"]:
    print(f"  {k:20s} {len(f['mask'][k]):4d} demos")

assert len(demos) == 200
assert len(f["mask/train"]) == 180 and len(f["mask/valid"]) == 20

`train`/`valid` is the split we will train on. `20_percent` / `50_percent` are pre-made
**demonstration-budget subsets**: exactly what phase 3 needs ("how many demos does a method need?").

### What is inside one demo?

`num_samples` = number of **transitions** = number of policy decisions in this episode.
Every array in the demo has this as its first dimension, like the frame axis of a video.

In [ ]:
g = f["data/demo_0"]
print("demo_0 attrs:", {k: v for k, v in g.attrs.items() if k != "model_file"},
      "| model_file: MuJoCo XML,", len(g.attrs["model_file"]), "chars")
g.visititems(lambda name, obj: print(f"  {name:32s} {obj.shape}") if isinstance(obj, h5py.Dataset) else None)

with h5py.File(RAW, "r") as fr:
    print("\nraw file (demo_v15.hdf5), demo_0 keys:", list(fr["data/demo_0"].keys()))

The **raw** file has no `obs`: only `states`, `actions` and some collection metadata.
`low_dim_v15.hdf5` was produced from it by `dataset_states_to_obs.py`, which loads each state
into the simulator and records observations. Raw states are the source of truth; obs (and
later camera images) are derived from them.

## 2. Time: what "20 Hz" means

`control_freq: 20` means the policy issues **one action every 1/20 s = 0.05 s**. One row of
`actions` is one such decision. The first entry of every state row is simulation time,
so we can check it directly.

In [ ]:
env_args = json.loads(f["data"].attrs["env_args"])
kw = env_args["env_kwargs"]
ctrl = kw["controller_configs"]["body_parts"]["right"]
control_freq = kw["control_freq"]
print("env:", env_args["env_name"], env_args["env_version"], "| control_freq:", control_freq, "Hz")

t = g["states"][:, 0]
dt = np.diff(t)
print("states[:, 0] (time):", np.round(t[:5], 3), "... | dt:", np.unique(np.round(dt, 6)))
assert np.allclose(dt, 1 / control_freq)

MODEL_TIMESTEP = 0.002  # robosuite macros.SIMULATION_TIMESTEP (500 Hz physics)
print(f"physics substeps per action: {round(1 / control_freq / MODEL_TIMESTEP)}")

In [ ]:
lengths = np.array([f[f"data/{d}"].attrs["num_samples"] for d in demos])
print(f"steps per demo: min {lengths.min()}, mean {lengths.mean():.1f}, max {lengths.max()}")
print(f"seconds per demo: {lengths.min() / control_freq:.1f} - {lengths.max() / control_freq:.1f}")

fig, ax = plt.subplots(figsize=(6, 2.5))
ax.hist(lengths, bins=15)
ax.set(xlabel="num_samples (policy steps)", ylabel="# demos",
       title=f"Lift ph episode length ({control_freq} Hz -> step = {1000 // control_freq} ms)")
plt.tight_layout()

## 3. Actions: commands to a controller, not target answers

From `env_args`, each action is 7 numbers in `[-1, 1]`:

```
[dx, dy, dz,  dax, day, daz,  gripper]
 position delta  rotation delta    -1 open / +1 close
                 (axis-angle)
```

`control_delta: true` means relative to the current end-effector pose; `input_ref_frame:
world` means expressed in world axes. The controller rescales `[-1, 1]` to
`output_min/max`: up to 5 cm and 0.5 rad per step.

In [ ]:
print("controller:", ctrl["type"], "| delta:", ctrl["control_delta"], "| frame:", ctrl["input_ref_frame"],
      "| kp:", ctrl["kp"], "| damping:", ctrl["damping"])
print("output_max:", ctrl["output_max"], "| gripper:", ctrl["gripper"]["type"])
SCALE = np.array(ctrl["output_max"])  # [-1, 1] -> metres / radians

all_actions = np.concatenate([f[f"data/{d}/actions"][:] for d in demos])
print("action min:", all_actions.min(0).round(2))
print("action max:", all_actions.max(0).round(2))
print("gripper values:", np.unique(all_actions[:, 6]))
assert np.all(np.abs(all_actions) <= 1 + 1e-6)
assert set(np.unique(all_actions[:, 6])) <= {-1.0, 1.0}

In [ ]:
DIM_NAMES = ["dx", "dy", "dz", "dax", "day", "daz", "gripper"]

def plot_actions(demo, ax_pos, ax_rot, ax_grip, alpha=1.0, label=True):
    a = f[f"data/{demo}/actions"][:]
    ts = np.arange(len(a)) / control_freq
    for i, c in zip(range(3), ["C0", "C1", "C2"]):
        ax_pos.plot(ts, a[:, i], c, alpha=alpha, label=DIM_NAMES[i] if label else None)
        ax_rot.plot(ts, a[:, 3 + i], c, alpha=alpha, label=DIM_NAMES[3 + i] if label else None)
    ax_grip.step(ts, a[:, 6], "k", where="post", alpha=alpha)

fig, axes = plt.subplots(3, 1, figsize=(8, 6), sharex=True)
plot_actions("demo_0", *axes)
axes[0].set_ylabel("position cmd"); axes[1].set_ylabel("rotation cmd"); axes[2].set_ylabel("gripper")
axes[0].legend(ncol=3, fontsize=8); axes[1].legend(ncol=3, fontsize=8)
axes[2].set_xlabel("time [s]")
fig.suptitle("demo_0 actions: approach (x, z<0) -> close gripper -> lift (z>0)")
plt.tight_layout()

**Look for:** the phases of the task. Moving towards the cube (`dx`, `dz < 0`), the gripper flip
from -1 to +1, then a large `dz > 0` to lift. Rotation commands stay small: Lift barely needs
reorientation.

Now overlay several demos. Same task, different human timing and paths: this spread is the
"label noise" a policy must model.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(8, 6), sharex=True)
for i, d in enumerate(demos[:8]):
    plot_actions(d, *axes, alpha=0.6, label=(i == 0))
axes[0].set_ylabel("position cmd"); axes[1].set_ylabel("rotation cmd"); axes[2].set_ylabel("gripper")
axes[2].set_xlabel("time [s]"); fig.suptitle("8 demos overlaid")
plt.tight_layout()

### Where did the hand actually go? (3D)

The curves above are **commands**, not positions. For the actual path we do not need to integrate
anything: `obs/robot0_eef_pos` already records the absolute hand position at every step (what
MuJoCo produced). For comparison, the dashed line naively adds up the commanded deltas
(`cumsum(action[:, :3] * 0.05 m)`), which is *not* how the robot moves: it ends far away and even below the table.
The cube sits on the table at z ≈ 0.83 m; the lift is only a few centimetres.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers the 3D projection)

def demo_path(demo):
    gd = f[f"data/{demo}"]
    pos = gd["obs/robot0_eef_pos"][:]
    cube0 = gd["obs/object"][0, :3]
    close_t = int(np.argmax(gd["actions"][:, 6] > 0))
    naive = pos[0] + np.cumsum(gd["actions"][:, :3] * SCALE[:3], axis=0)
    return pos, cube0, close_t, naive

pos, cube0, close_t, naive = demo_path("demo_0")
fig = plt.figure(figsize=(12, 4.5))

ax = fig.add_subplot(1, 3, 1, projection="3d")
ax.plot(*pos[:close_t].T, "C0.-", ms=3, label="hand, gripper open")
ax.plot(*pos[close_t:].T, "C3.-", ms=3, label="hand, gripper closing/closed")
ax.scatter(*cube0, s=80, c="C2", marker="s", label="cube at start")
ax.scatter(*pos[0], s=40, c="k", label="start")
ax.set(xlabel="x [m]", ylabel="y [m]", zlabel="z [m]", title="demo_0 in 3D")
ax.legend(fontsize=7, loc="upper left")

ax = fig.add_subplot(1, 3, 2)
ax.plot(pos[:close_t, 0], pos[:close_t, 2], "C0.-", ms=3)
ax.plot(pos[close_t:, 0], pos[close_t:, 2], "C3.-", ms=3)
ax.plot(naive[:, 0], naive[:, 2], "k--", lw=0.8, label="naive sum of commands")
ax.axhline(cube0[2] - 0.02, color="0.6", lw=0.8, label="table top (approx.)")
ax.legend(fontsize=7)
ax.scatter(cube0[0], cube0[2], s=80, c="C2", marker="s")
ax.set(xlabel="x [m]", ylabel="z (height) [m]", title="side view (x-z): actual vs naive sum")
ax.set_aspect("equal", adjustable="datalim")

ax = fig.add_subplot(1, 3, 3, projection="3d")
for i, d in enumerate(demos[:10]):
    p, c0, ct, _ = demo_path(d)
    ax.plot(*p.T, color=f"C{i}", lw=1)
    ax.scatter(*c0, s=30, color=f"C{i}", marker="s")
ax.set(xlabel="x", ylabel="y", zlabel="z", title="10 demos: paths and cube start (squares)")
plt.tight_layout()

err = np.linalg.norm(naive[-1] - pos[-1])
print(f"demo_0 final hand position: actual {pos[-1].round(3)}, naive sum {naive[-1].round(3)}, gap {err * 100:.1f} cm")

### What happens inside `env.step(action)`

```
policy (20 Hz) --action--> OSC controller (robosuite, Python) --joint torques--> MuJoCo (500 Hz) --> new state
                           recomputed at every one of the 25 physics substeps,
                           using MuJoCo's Jacobian and mass matrix
```

The controller acts like a spring-damper (`kp`, `damping`) pulling the hand towards
"current pose + delta". It does **not** teleport the hand there in one step. We can measure that:
commanded displacement (`action * 0.05 m`) vs the actual end-effector displacement in the same step.

In [ ]:
cmd, real = [], []
for d in demos:
    gd = f[f"data/{d}"]
    cmd.append(gd["actions"][:, :3] * SCALE[:3])
    real.append(gd["next_obs/robot0_eef_pos"][:] - gd["obs/robot0_eef_pos"][:])
cmd, real = np.concatenate(cmd), np.concatenate(real)

fig, axes = plt.subplots(1, 3, figsize=(10, 3), sharey=True)
for i, ax in enumerate(axes):
    slope = np.polyfit(cmd[:, i], real[:, i], 1)[0]
    r = np.corrcoef(cmd[:, i], real[:, i])[0, 1]
    ax.scatter(cmd[:, i] * 100, real[:, i] * 100, s=1, alpha=0.2)
    lim = 5
    ax.plot([-lim, lim], [-lim, lim], "k--", lw=0.8, label="real = cmd")
    ax.set(title=f"{'xyz'[i]}: slope {slope:.2f}, r {r:.2f}", xlabel="commanded [cm/step]")
axes[0].set_ylabel("actual [cm/step]"); axes[0].legend(fontsize=8)
plt.tight_layout()

**Reading the plot:** strongly correlated, but the hand moves only a fraction of the commanded
delta within one step, and the scatter depends on velocity and contacts. An action is an
*instruction to a dynamic system*, not a label of where the hand will be. This is also why
replaying recorded actions under a different MuJoCo version can drift.

## 4. `states`: the full simulator snapshot (`qpos`, `qvel`)

MuJoCo describes the world with **generalized coordinates**:
- `qpos`: positions of every joint. A hinge (arm joint) adds 1 angle; a slide (finger) adds 1
  distance; a **free joint** (the loose cube) adds 7 = xyz + quaternion.
- `qvel`: their velocities. Same joints, but a free joint has 6 (linear + angular velocity),
  so `len(qvel)` can differ from `len(qpos)`.

We rebuild the layout from the scene XML stored in `model_file`.

In [ ]:
QPOS_DIM = {"hinge": 1, "slide": 1, "ball": 4, "free": 7}
QVEL_DIM = {"hinge": 1, "slide": 1, "ball": 3, "free": 6}

scene = ET.fromstring(g.attrs["model_file"])
joints = [(j.get("name"), "free" if j.tag == "freejoint" else j.get("type", "hinge"))
          for j in scene.iter() if j.tag in ("joint", "freejoint")]
nq = sum(QPOS_DIM[t] for _, t in joints)
nv = sum(QVEL_DIM[t] for _, t in joints)
for name, jt in joints:
    print(f"  {name:30s} {jt:6s} qpos {QPOS_DIM[jt]}  qvel {QVEL_DIM[jt]}")
print(f"time 1 + qpos {nq} + qvel {nv} = {1 + nq + nv}  vs states width {g['states'].shape[1]}")
assert 1 + nq + nv == g["states"].shape[1]

In [ ]:
s, o = g["states"][:], g["obs"]
qpos = s[:, 1:1 + nq]
checks = {
    "arm joints   qpos[0:7]   == obs robot0_joint_pos": np.allclose(qpos[:, 0:7], o["robot0_joint_pos"][:]),
    "fingers      qpos[7:9]   == obs robot0_gripper_qpos": np.allclose(qpos[:, 7:9], o["robot0_gripper_qpos"][:]),
    "cube xyz     qpos[9:12]  == obs object[0:3]": np.allclose(qpos[:, 9:12], o["object"][:, 0:3]),
    "cube quat    qpos[12:16] (w,x,y,z) == object[3:7] (x,y,z,w) reordered":
        np.allclose(qpos[:, [13, 14, 15, 12]], o["object"][:, 3:7]),
}
for k, v in checks.items():
    print(f"{'OK ' if v else 'NO '} {k}")
assert all(checks.values())

Two lessons:
- **obs are computed from state.** At rollout time the simulator returns the same obs keys from
  `env.step()`; the policy never sees `states`. `states` serve **reset and replay**: put the
  simulator exactly where the demo was.
- **Quaternion convention trap:** MuJoCo stores `(w, x, y, z)`, robosuite obs use `(x, y, z, w)`.
  Mixing them silently breaks any code that touches orientations.

## 5. `obs` vs `next_obs`, `rewards`, `dones`

These follow the RL transition tuple `(s_t, a_t, r_t, s_{t+1}, done_t)`. Behaviour cloning uses
only `obs -> actions`; the rest is kept for offline RL algorithms.

In [ ]:
for k in o:
    assert np.allclose(g[f"next_obs/{k}"][:-1], o[k][1:]), k
print("next_obs[t] == obs[t+1] for all", len(o), "keys")

r, dn = g["rewards"][:], g["dones"][:]
print("rewards:", r.astype(int))
print("first success step:", int(np.argmax(r > 0)), "of", len(r))

first_success = np.array([np.argmax(f[f"data/{d}/rewards"][:] > 0) for d in demos])
print("steps recorded after first success: min", (lengths - first_success).min(),
      "max", (lengths - first_success).max())

Reward is **sparse** (`reward_shaping: false`): 1 when the cube is lifted above a height
threshold, else 0. Demos stop a few steps after success. During our own evaluation the env
does not stop by itself (`ignore_done: true`), so the evaluator decides success and horizon.

## 6. Multimodality you can see: when to close the gripper

The gripper action is binary. Plot P(close) against the gripper-to-cube distance (last 3
numbers of `object`). If humans always closed at the same distance, this would be a sharp step.

In [ ]:
dist = np.concatenate([np.linalg.norm(f[f"data/{d}/obs/object"][:, 7:10], axis=1) for d in demos])
grip = all_actions[:, 6]
bins = np.linspace(0, 0.06, 13)
idx = np.digitize(dist, bins) - 1
centers = (bins[:-1] + bins[1:]) / 2
p_close = [np.mean(grip[idx == i] > 0) if np.any(idx == i) else np.nan for i in range(len(centers))]
counts = [int(np.sum(idx == i)) for i in range(len(centers))]

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(centers * 100, p_close, "o-")
ax.set(xlabel="gripper-to-cube distance [cm]", ylabel="P(gripper = close)", ylim=(-0.05, 1.05),
       title="Same situation, different expert choices")
for c, p, n in zip(centers, p_close, counts):
    if n: ax.annotate(str(n), (c * 100, p), textcoords="offset points", xytext=(0, 6), fontsize=7, ha="center")
plt.tight_layout()

Between roughly 1.5 and 3 cm the experts disagree: some already close, some wait. The target
for that input is two-peaked (-1 or +1). A regression head trained with MSE predicts the
mean, a half-closed gripper that no expert ever commanded. This is the reason for GMM heads
(BC-RNN), Diffusion Policy and ACT's latent variable.

Note that the cube is **not** at the same place in every demo: object positions are randomized at
reset, which is why `object` is part of the observation.

## 7. Inside one `env.step`: replay demo_0 and record the torques

Sections 1-6 only read the file. Here we run the simulator (CPU, no rendering, a few seconds):
rebuild the env from `env_args`, reset to the demo's first state, re-execute its 59 actions, and
record what the arm receives at **every physics substep** (`sim.data.ctrl`).

Expected: 59 actions x 25 substeps = 1475 torque vectors of 7 numbers (N·m). The OSC goal is set
only at the first substep of each action (`policy_step=True`); the torque is recomputed at all 25.
Wrapping `robot.control` is an inspection trick only, not code to reuse.

In [ ]:
import robomimic.utils.env_utils as EnvUtils
import robomimic.utils.obs_utils as ObsUtils

ObsUtils.initialize_obs_modality_mapping_from_dict({"low_dim": list(o.keys())})
env = EnvUtils.create_env_from_metadata(env_args, render=False, render_offscreen=False, use_image_obs=False)
env.reset()
env.reset_to({"states": g["states"][0], "model": g.attrs["model_file"]})

robot = env.env.robots[0]
model = robot.sim.model
actuators = [model.actuator_id2name(i) for i in range(model.nu)]
print("actuators (sim.data.ctrl):", actuators)
print("ctrlrange:", model.actuator_ctrlrange.tolist())

log = []  # one row per physics substep: (policy_step, sim time, ctrl vector)
_control = robot.control
def recording_control(action, policy_step=False):
    _control(action, policy_step=policy_step)
    log.append((policy_step, robot.sim.data.time, robot.sim.data.ctrl.copy()))
robot.control = recording_control

demo_actions, demo_states = g["actions"][:], g["states"][:]
replay_states, replay_success = [], []
for a in demo_actions:
    env.step(a)
    replay_states.append(env.get_state()["states"])
    replay_success.append(env.is_success()["task"])
robot.control = _control

is_policy_step = np.array([p for p, _, _ in log])
sim_t = np.array([t for _, t, _ in log])
ctrl_log = np.stack([c for _, _, c in log])
print(f"substeps recorded: {len(log)} = {is_policy_step.sum()} actions x {len(log) // is_policy_step.sum()}")
print("ctrl_log shape:", ctrl_log.shape, "-> arm torques", ctrl_log[:, :7].shape, "+ finger targets", ctrl_log[:, 7:].shape)
assert len(log) == len(demo_actions) * round(1 / control_freq / MODEL_TIMESTEP)

In [ ]:
arm = ctrl_log[:, :7]
fig, axes = plt.subplots(3, 1, figsize=(9, 7), gridspec_kw={"height_ratios": [2, 2, 1]})

for j in range(7):
    axes[0].plot(sim_t, arm[:, j], lw=0.8, label=f"j{j + 1}")
axes[0].set(ylabel="torque [N·m]", title="Arm torques for all of demo_0 (one point every 2 ms)")
axes[0].legend(ncol=7, fontsize=7)

k0, k1 = 20, 24  # zoom into 4 consecutive actions
win = (sim_t >= k0 / control_freq) & (sim_t < k1 / control_freq)
for j in range(7):
    axes[1].plot(sim_t[win] * 1000, arm[win, j], ".-", ms=2, lw=0.8)
for k in range(k0, k1 + 1):
    axes[1].axvline(k * 1000 / control_freq, color="k", lw=0.6, ls="--")
axes[1].set(ylabel="torque [N·m]", xlabel="time [ms]",
            title=f"Zoom: actions {k0}-{k1 - 1}; dashed line = new action, 25 torque vectors between lines")

axes[2].plot(sim_t, ctrl_log[:, 7], label="finger 1 target")
axes[2].plot(sim_t, ctrl_log[:, 8], label="finger 2 target")
axes[2].set(ylabel="position [m]", xlabel="time [s]", title="Gripper: position targets, not torques")
axes[2].legend(fontsize=7)
plt.tight_layout()

**Look for:**
- In the zoom, torques jump at each dashed line (new goal) and then change smoothly within the
  action as the arm moves and the error shrinks. One action -> a `(25, 7)` torque sequence.
- Large, nearly constant torque on joint 2 (shoulder): mostly gravity compensation.
- The finger targets ramp instead of jumping: each action moves the gripper command by 0.2 of
  its [-1, 1] range, so fully open -> closed takes about 10 actions (0.5 s). MuJoCo's position
  actuators then produce the finger force.

The dataset stores none of this; it keeps only the 20 Hz layer (`states`, `actions`).

In [ ]:
replay_states = np.array(replay_states)
first_recorded = int(np.argmax(g["rewards"][:] > 0))
first_replayed = int(np.argmax(replay_success))
print(f"first success: recorded step {first_recorded}, replayed step {first_replayed}, "
      f"success at end of replay: {replay_success[-1]}")

qpos_err = np.abs(replay_states[:-1, 1:1 + nq] - demo_states[1:, 1:1 + nq]).max(axis=1)
fig, ax = plt.subplots(figsize=(7, 2.5))
ax.semilogy(np.arange(1, len(demo_states)) / control_freq, qpos_err)
ax.axvline(first_recorded / control_freq, color="k", ls="--", lw=0.8, label="recorded success")
ax.set(xlabel="time [s]", ylabel="max |qpos error|", title="Open-loop replay vs recorded states")
ax.legend(fontsize=8)
plt.tight_layout()

Replaying recorded actions open-loop reproduces the demo closely but not exactly: the error is
tiny while the arm moves in free space and grows once the fingers touch the cube (contact is
sensitive to tiny differences). Whether it still **succeeds** is what matters. Step 0.1d checks
this for all demos and renders videos with the upstream `playback_dataset.py`.

## 8. Your notes

Write 3-5 bullet points in your own words before moving on to replay (0.1d):
- What exactly is one row of `actions`, physically?
A row of actions is a teleoperator's command (which the policy will imitate) according to the observations in a particular time step. Physically, an action gives a targeted delta of the end-effector pose, which contains the translational coordinate change, axis-angle change, and the open/close of the gripper. It set a new goal at 20 HZ frequency, while the OSC translate the goal to physical torque, get simulated feedbacks from MuJoCo, and update observations. In our environment, the OSC reach ~22% of the action's command (evidence: S3).
- Why can't we evaluate this policy by validation loss alone?
Low MSE loss only means the model fit the training demo data, which only requires the model to predict the actions as close to the demo for a given observation. The success of an episode however, requires consecutive actions, which involves multiple turns of obs and action cycle.
Val loss is only measured on the demo states, but in real-world inference and rollout, small errors can push thee robot into unseen states, which can compound errors to the later time steps. Also, MSE loss does not prioritise critical steps, such as grasping object, selecting checkpoints by validation loss proven to have little correlation to the success rate (by robomimic paper).
- Which obs keys would you give a policy, and which are redundant?
Policy needs position, rotations of robot itself, and of the target. Here:
1. robot0_joint_pos, robot0_joint_pos_cos, robot0_joint_pos_sin all provide position of the robot joints, just with different representation. 
2. robot0_eef_pos, robot0_eef_quat, robot0_eef_quat_site, on the other hand, provide position of end-effector(eef), and can be induced from 1. using forward-kinematics (fk). Choose one key from 1. or 2. would be enough for this task.
3. robot0_gripper_qpos and robot0_gripper_qvel gives the position and velocity of the eef grip, respectively.
4. object gives the position and rotation of the object, where object[0:3] shows position, object[3:7] gives quantonion rotation, and object[7:10] = object[0:3] - eef_pos is just a feature representing the distance between object and eef.

In [ ]:
f.close()